# Feature Engineering

In [1]:
import warnings
warnings.filterwarnings('ignore')

import re
import math
import numpy as np
import pandas as pd
from tqdm import tqdm
from collections import Counter
from datasets import load_dataset

from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from rank_bm25 import BM25Okapi
import pickle

## Data Ingestion

In [2]:
dataset = load_dataset("ms_marco", "v2.1", split="validation")
marco_df = dataset.to_pandas()
marco_df.head()

,answers,passages,query,query_id,query_type,wellFormedAnswers
0,[A corporation is a company or group of people...,"{'is_selected': [0, 0, 0, 0, 0, 1, 0, 0, 0, 0]...",. what is a corporation?,1102432,DESCRIPTION,[]
1,[Rachel Carson writes The Obligation to Endure...,"{'is_selected': [0, 0, 0, 0, 1, 1, 0, 0, 0, 0]...",why did rachel carson write an obligation to e...,1102431,DESCRIPTION,[]
2,[No Answer Present.],"{'is_selected': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0]...",why did the progressive movement fail to advan...,1102421,DESCRIPTION,[]
3,[No Answer Present.],"{'is_selected': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0]...",why do police need to understand what the fore...,1102315,DESCRIPTION,[]
4,[No Answer Present.],"{'is_selected': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0]...",do owls eat in the day,1101280,NUMERIC,[]


| Column | Type | Notes |
| :--- | :--- | :--- |
| query | string | The search query |
| passages | dict | Contains passage_text (list) and is_selected (list) |
| query_id | int | Unique query identifier |

In [3]:
marco_df.shape

(101093, 6)

Current size might be a bit too large for development purposes. Would be sub-sampling to 10,000 rows using random_state of 42 to ensure reproducibility

In [4]:
marco_df_samp = marco_df.sample(n=10000, random_state=42).reset_index(drop=True)

## Data Pre-processing

### Explode Rows

In [5]:
marco_df.columns

Index(['answers', 'passages', 'query', 'query_id', 'query_type',
       'wellFormedAnswers'],
      dtype='str')

In [6]:
marco_df_exp = marco_df_samp.copy()
marco_df_exp['passage_text'] = marco_df_samp['passages'].apply(lambda x: x['passage_text'])
marco_df_exp['is_selected'] = marco_df_samp['passages'].apply(lambda x: x['is_selected'])

marco_df_exp = marco_df_exp.explode(['passage_text', 'is_selected']).reset_index(drop=True)
marco_df_exp['label'] = marco_df_exp['is_selected'].astype(int)
print(marco_df_exp.shape) 

(99840, 9)


In [7]:
valid_query_ids = marco_df_exp.groupby('query_id')['is_selected'].sum()
valid_query_ids = valid_query_ids[valid_query_ids > 0].index
marco_df_exp = marco_df_exp[marco_df_exp['query_id'].isin(valid_query_ids)]

print(f"Remaining rows: {marco_df_exp.shape[0]:,}")
print(f"Unique queries: {marco_df_exp['query_id'].nunique():,}")

Remaining rows: 54,281
Unique queries: 5,435


## Feature Engineering

#### Lexical Matching Features
*"Do the query words actually appear in the passage?"*

| Feature | Why It Matters |
| :--- | :--- |
| BM25 score | The gold standard sparse retrieval signal, what Bing/Google used before neural models |
| TF-IDF cosine similarity | Measures vocabulary overlap between query and passage |
| Query term coverage | % of query words found in the passage |
| Exact match flag | Binary - does the full query appear verbatim in the passage? |

#### Semantic Features
*"Do they mean the same thing, even with different words?"*
| Feature | Why It Matters |
| :--- | :--- |
| TF-IDF vector similarity | Captures related vocabulary beyond exact matches |
| Query-passage Jaccard similarity | Simple word overlap ratio |

We may substitute TF-IDF for BERT depending on computing power

#### Statistical / Length Features
*"What does the structure of the passage tell us?"*

| Feature | Why It Matters |
| :--- | :--- |
| Passage length (word count) | Longer passages may dilute relevance |
| Query length (word count) | Longer queries are more specific |
| Length ratio | query length / passage length |
| Passage position | Earlier passages in the original list tend to be more relevant |

#### Label 

| Column | Role |
| :--- | :--- |
| is_selected | Our binary target where 1 = relevant, 0 = not relevant |




#### Query Length, Passage Length, Length Ratio

In [8]:
marco_df_exp['query_length'] = marco_df_exp['query'].str.split().str.len()
marco_df_exp['passage_length'] = marco_df_exp['passage_text'].str.split().str.len()
marco_df_exp['length_ratio'] = marco_df_exp['query_length'] / (marco_df_exp['passage_length'] + 1)
print(marco_df_exp[['query_length', 'passage_length', 'length_ratio']].describe())

       query_length  passage_length  length_ratio
count  54281.000000    54281.000000  54281.000000
mean       5.907574       51.209889      0.126205
std        2.540894       18.997756      0.074595
min        2.000000        1.000000      0.014286
25%        4.000000       40.000000      0.078431
50%        6.000000       48.000000      0.111111
75%        7.000000       57.000000      0.153846
max       30.000000      196.000000      2.500000


#### Exact match flag, Query term coverage

In [9]:
def exact_match(query, passage):
    return int(str(query).lower() in str(passage).lower())

def query_term_coverage(query, passage):
    query_words = set(str(query).lower().split())
    passage_words = set(str(passage).lower().split())
    if len(query_words) == 0:
        return 0.0
    return len(query_words & passage_words) / len(query_words)

In [10]:
marco_df_exp['exact_match'] = marco_df_exp.apply(
    lambda row: exact_match(row['query'], row['passage_text']), axis=1
)

marco_df_exp['query_term_coverage'] = marco_df_exp.apply(
    lambda row: query_term_coverage(row['query'], row['passage_text']), axis=1
)

In [11]:
print(marco_df_exp[['exact_match', 'query_term_coverage']].describe())

        exact_match  query_term_coverage
count  54281.000000         54281.000000
mean       0.010464             0.441115
std        0.101758             0.240172
min        0.000000             0.000000
25%        0.000000             0.272727
50%        0.000000             0.428571
75%        0.000000             0.600000
max        1.000000             1.000000


#### Jaccard Similarity

$$J(A, B) = \frac{|A \cap B|}{|A \cup B|}$$

In [12]:
def jaccard_similarity(query, passage):
    query_words = set(str(query).lower().split())
    passage_words = set(str(passage).lower().split())
    if len(query_words | passage_words) == 0:
        return 0.0
    return len(query_words & passage_words) / len(query_words | passage_words)

In [13]:
marco_df_exp['jaccard_similarity'] = marco_df_exp.apply(
    lambda row: jaccard_similarity(row['query'], row['passage_text']), axis=1
)

In [14]:
print(marco_df_exp[['jaccard_similarity']].describe())

       jaccard_similarity
count        54281.000000
mean             0.063802
std              0.046048
min              0.000000
25%              0.031250
50%              0.056818
75%              0.085714
max              1.000000


#### TF-IDF Similarity

- TF-IDF Weighting
$$w_{t,d} = \text{tf}_{t,d} \times \log\left(\frac{N}{\text{df}_t}\right)$$

- Cosine Similarity (The "Similarity" Part)
$$\text{similarity}(A, B) = \frac{A \cdot B}{\|A\| \|B\|} = \frac{\sum_{i=1}^{n} A_i B_i}{\sqrt{\sum_{i=1}^{n} A_i^2} \sqrt{\sum_{i=1}^{n} B_i^2}}$$

In [15]:
all_text = pd.concat([marco_df_exp['query'], marco_df_exp['passage_text']], ignore_index=True)
vectorizer = TfidfVectorizer(max_features=50000, ngram_range=(1,2), sublinear_tf=True)
vectorizer.fit(all_text)

BATCH_SIZE = 5000
tfidf_scores = []

for i in tqdm(range(0, len(marco_df_exp), BATCH_SIZE), desc="TF-IDF batches"):
    batch = marco_df_exp.iloc[i:i+BATCH_SIZE]
    query_vecs = vectorizer.transform(batch['query'])
    passage_vecs = vectorizer.transform(batch['passage_text'])
    scores = cosine_similarity(query_vecs, passage_vecs).diagonal()
    tfidf_scores.extend(scores)

marco_df_exp['tfidf_cosine_sim'] = tfidf_scores


TF-IDF batches: 100%|██████████| 11/11 [00:05<00:00,  1.93it/s]


In [16]:
marco_df_exp['tfidf_cosine_sim'].describe()

count    54281.000000
mean         0.194105
std          0.139623
min          0.000000
25%          0.088628
50%          0.174015
75%          0.279113
max          1.000000
Name: tfidf_cosine_sim, dtype: float64

In [17]:
with open('tfidf_vectorizer.pkl', 'wb') as f:
    pickle.dump(vectorizer, f)

#### BM25

BM25 is a bag-of-words retrieval function that ranks a set of documents based on the query terms appearing in each document, regardless of their proximity within the document. It is a family of scoring functions with slightly different components and parameters. One of the most prominent instantiations of the function is as follows.

$$\text{score}(D, Q) = \sum_{i=1}^{n} \text{IDF}(q_i) \cdot \frac{f(q_i, D) \cdot (k_1 + 1)}{f(q_i, D) + k_1 \cdot \left(1 - b + b \cdot \frac{|D|}{\text{avgdl}}\right)}$$


- $f(q_i, D)$: Number of times the query term $q_i$ occurs in document $D$
- $|D|$: Length of document $D$ (in words)
- $\text{avgdl}$: Average document length in the collection
- $k_1$: Term frequency saturation parameter  
  - Typically $k_1 \in [1.2, 2.0]$
- $b$: Length normalization parameter  
  - Commonly set to $b = 0.75$


The inverse document frequency weight for a query term \( q_i \) is usually computed as:
$$\text{IDF}(q_i) = \ln \left( \frac{N - n(q_i) + 0.5}{n(q_i) + 0.5} + 1 \right)$$

where:
- $(N)$: Total number of documents in the collection
- $(n(q_i))$: Number of documents containing term $(q_i)$

In [18]:

print("Tokenizing passages...")
tokenized_passages = [str(p).lower().split() for p in marco_df_exp['passage_text']]
bm25 = BM25Okapi(tokenized_passages)
print(f"BM25 index built over {len(tokenized_passages):,} passages")

# Get unique queries
unique_queries = marco_df_exp[['query_id', 'query']].drop_duplicates()
print(f"Scoring {len(unique_queries):,} unique queries...")

# Score each unique query once
bm25_score_map = {}
for _, row in tqdm(unique_queries.iterrows(), total=len(unique_queries), desc="BM25 scoring"):
    query_tokens = str(row['query']).lower().split()
    bm25_score_map[row['query_id']] = bm25.get_scores(query_tokens)

# Map scores back
marco_df_exp = marco_df_exp.reset_index(drop=True)
marco_df_exp['bm25_score'] = [
    bm25_score_map[row['query_id']][idx]
    for idx, row in marco_df_exp.iterrows()
]


Tokenizing passages...
BM25 index built over 54,281 passages
Scoring 5,435 unique queries...


BM25 scoring: 100%|██████████| 5435/5435 [10:19<00:00,  8.77it/s]


In [19]:
with open('bm25_scores.pkl', 'wb') as f:
    pickle.dump(marco_df_exp['bm25_score'].tolist(), f)

with open('bm25_index.pkl', 'wb') as f:
    pickle.dump(bm25, f)

print(f"BM25 done — saved to bm25_scores.pkl and bm25_index.pkl")
print(marco_df_exp['bm25_score'].describe().round(3))

BM25 done — saved to bm25_scores.pkl and bm25_index.pkl
count    54281.000
mean        15.491
std         10.302
min          0.000
25%          8.263
50%         14.647
75%         21.594
max        119.100
Name: bm25_score, dtype: float64


#### Passage Position

In [20]:
marco_df_exp['passage_position'] = marco_df_exp.groupby('query_id').cumcount()

## Enriched Features

### IDF Weighted Coverage

In [21]:
with open('tfidf_vectorizer.pkl', 'rb') as f:
    vectorizer = pickle.load(f)

In [22]:
def idf_weighted_coverage(query, passage, vectorizer):
    query_words = str(query).lower().split()
    passage_words = set(str(passage).lower().split())
    vocab = vectorizer.vocabulary_
    idf = vectorizer.idf_

    score = 0
    total = 0
    for word in query_words:
        if word in vocab:
            w = idf[vocab[word]]
            score += w if word in passage_words else 0
            total += w
    return score / total if total > 0 else 0.0

tqdm.pandas(desc="IDF-weighted coverage")
marco_df_exp['idf_weighted_coverage'] = marco_df_exp.progress_apply(
    lambda row: idf_weighted_coverage(row['query'], row['passage_text'], vectorizer), axis=1
)
print("Enriched Feature 1 done: idf_weighted_coverage")


IDF-weighted coverage: 100%|██████████| 54281/54281 [00:01<00:00, 39192.06it/s]

Enriched Feature 1 done: idf_weighted_coverage


###  Bigram & Trigram Overlap

In [23]:
def ngram_overlap(query, passage, n=2):
    try:
        vec = CountVectorizer(ngram_range=(n, n), binary=True)
        matrix = vec.fit_transform([str(query).lower(), str(passage).lower()])
        q_ngrams = set(matrix[0].indices)
        p_ngrams = set(matrix[1].indices)
        if len(q_ngrams) == 0:
            return 0.0
        return len(q_ngrams & p_ngrams) / len(q_ngrams)
    except:
        return 0.0

tqdm.pandas(desc="Bigram overlap")
marco_df_exp['bigram_overlap'] = marco_df_exp.progress_apply(
    lambda row: ngram_overlap(row['query'], row['passage_text'], n=2), axis=1
)

tqdm.pandas(desc="Trigram overlap")
marco_df_exp['trigram_overlap'] = marco_df_exp.progress_apply(
    lambda row: ngram_overlap(row['query'], row['passage_text'], n=3), axis=1
)
print("Feature 2 done: bigram_overlap, trigram_overlap")

Trigram overlap: 100%|██████████| 54281/54281 [00:23<00:00, 2264.86it/s]

Feature 2 done: bigram_overlap, trigram_overlap


### Number of Features + Query Type Interactions

In [24]:
marco_df_exp['passage_has_number'] = marco_df_exp['passage_text'].apply(
    lambda x: int(bool(re.search(r'\d+', str(x))))
)
marco_df_exp['query_has_number'] = marco_df_exp['query'].apply(
    lambda x: int(bool(re.search(r'\d+', str(x))))
)
marco_df_exp['both_have_number'] = (
    marco_df_exp['passage_has_number'] & marco_df_exp['query_has_number']
).astype(int)

# Query type interaction features
marco_df_exp['is_numeric']     = (marco_df_exp['query_type'] == 'NUMERIC').astype(int)
marco_df_exp['is_description'] = (marco_df_exp['query_type'] == 'DESCRIPTION').astype(int)
marco_df_exp['is_entity']      = (marco_df_exp['query_type'] == 'ENTITY').astype(int)

# Cross features — DeepFM learns these interactions too, but explicit ones help
marco_df_exp['numeric_x_has_number']      = marco_df_exp['is_numeric'] * marco_df_exp['both_have_number']
marco_df_exp['description_x_length']      = marco_df_exp['is_description'] * marco_df_exp['passage_length']
marco_df_exp['entity_x_exact_match']      = marco_df_exp['is_entity'] * marco_df_exp['exact_match']

print("Feature 3 done: number features + query type interactions")

Feature 3 done: number features + query type interactions


### Max TF-IDF Score

In [25]:
def max_tfidf_term_score(query, passage, vectorizer):
    query_words = str(query).lower().split()
    vocab = vectorizer.vocabulary_
    idf = vectorizer.idf_

    try:
        passage_vec = vectorizer.transform([str(passage).lower()])
        scores = []
        for word in query_words:
            if word in vocab:
                scores.append(passage_vec[0, vocab[word]])
        return float(max(scores)) if scores else 0.0
    except:
        return 0.0

tqdm.pandas(desc="Max TF-IDF term score")
marco_df_exp['max_tfidf_term'] = marco_df_exp.progress_apply(
    lambda row: max_tfidf_term_score(row['query'], row['passage_text'], vectorizer), axis=1
)
print("Feature 4 done: max_tfidf_term")

Max TF-IDF term score: 100%|██████████| 54281/54281 [00:25<00:00, 2132.60it/s]

Feature 4 done: max_tfidf_term


### Final Check of Features

In [26]:
# feature_cols = [
#     'query_length', 'passage_length', 'length_ratio',
#     'exact_match', 'query_term_coverage', 'jaccard_similarity',
#     'tfidf_cosine_sim', 'bm25_score', 'passage_position'
# ]

feature_cols = [
    # Original features
    'query_length', 'passage_length', 'length_ratio',
    'exact_match', 'query_term_coverage', 'jaccard_similarity',
    'tfidf_cosine_sim', 'bm25_score', 'passage_position',
    # New features
    'idf_weighted_coverage',
    'bigram_overlap', 'trigram_overlap',
    'passage_has_number', 'query_has_number', 'both_have_number',
    'is_numeric', 'is_description', 'is_entity',
    'numeric_x_has_number', 'description_x_length', 'entity_x_exact_match',
    'max_tfidf_term'
]

In [27]:
print("Feature Summary:")
print(marco_df_exp[feature_cols].describe().round(3))

Feature Summary:
       query_length  passage_length  length_ratio  exact_match  \
count     54281.000       54281.000     54281.000    54281.000   
mean          5.908          51.210         0.126        0.010   
std           2.541          18.998         0.075        0.102   
min           2.000           1.000         0.014        0.000   
25%           4.000          40.000         0.078        0.000   
50%           6.000          48.000         0.111        0.000   
75%           7.000          57.000         0.154        0.000   
max          30.000         196.000         2.500        1.000   

       query_term_coverage  jaccard_similarity  tfidf_cosine_sim  bm25_score  \
count            54281.000           54281.000         54281.000   54281.000   
mean                 0.441               0.064             0.194      15.491   
std                  0.240               0.046             0.140      10.302   
min                  0.000               0.000             0.000    

In [28]:
print(f"\nShape: {marco_df_exp.shape}")


Shape: (54281, 31)


In [29]:
print(f"Missing values:\n{marco_df_exp[feature_cols].isnull().sum()}")

Missing values:
query_length             0
passage_length           0
length_ratio             0
exact_match              0
query_term_coverage      0
jaccard_similarity       0
tfidf_cosine_sim         0
bm25_score               0
passage_position         0
idf_weighted_coverage    0
bigram_overlap           0
trigram_overlap          0
passage_has_number       0
query_has_number         0
both_have_number         0
is_numeric               0
is_description           0
is_entity                0
numeric_x_has_number     0
description_x_length     0
entity_x_exact_match     0
max_tfidf_term           0
dtype: int64


In [30]:
# Save final dataframe
print(f"\Feature check:")
print(f"  Original features : 9")
print(f"  Total features    : {len(feature_cols)}")
# Save enriched dataframe
marco_df_exp.to_pickle('df_features_enriched.pkl')
print("\n Saved to df_features_enriched.pkl")

\Feature check:
  Original features : 9
  Total features    : 22

 Saved to df_features_enriched.pkl
